# Train Grasp-Anything++ trên Google Colab

Notebook này luôn đồng bộ code mới nhất từ branch `text-image-aware-v2`. Dataset, split, Hugging Face cache, log và checkpoint được lưu trên Google Drive. Sau khi dataset đã dựng xong, chạy lại notebook sẽ tái sử dụng cache và không download data lần nữa.

In [ ]:
#@title 1. Cấu hình — chỉnh mọi tham số chính tại đây
REPO_URL = "https://github.com/duncan-nguyen/QACI-HW.git"
BRANCH = "text-image-aware-v2"
DRIVE_ROOT = "/content/drive/MyDrive/[Research Space]/[QACI] VLA HW"  #@param {type:"string"}
LOCAL_REPO = "/content/QACI-HW"

# Data. 2.000 scene phù hợp để smoke train; tăng dần khi đã kiểm tra pipeline.
N_SCENES = 2000  #@param {type:"integer"}
PACK_MASKS = True  #@param {type:"boolean"}
IMAGES_FROM_ZIP = False  #@param {type:"boolean"}
KEEP_ARCHIVES = False  #@param {type:"boolean"}
DOWNLOAD_WORKERS = 16  #@param {type:"integer"}

# Train.
EPOCHS = 50  #@param {type:"integer"}
BATCHES_PER_EPOCH = 1000  #@param {type:"integer"}
# GPU ~90 GB VRAM: bắt đầu với 128; giảm còn 64 nếu gặp CUDA OOM.
BATCH_SIZE = 128  #@param {type:"integer"}
VAL_BATCH_SIZE = 256  #@param {type:"integer"}
NUM_WORKERS = 4  #@param {type:"integer"}
INPUT_SIZE = 224  #@param {type:"integer"}
VAL_SPLIT = 0.95  #@param {type:"number"}
LEARNING_RATE = 0.002  #@param {type:"number"}
DESCRIPTION = "ga-pp-colab"  #@param {type:"string"}

# Model text-image-aware-v2.
W_ALIGN = 0.3  #@param {type:"number"}
WARMUP_EPOCHS = 3  #@param {type:"integer"}
ALIGN_MODE = "soft"  #@param ["soft", "hard"]
REGION_TEXT = 1
FUSION = "residual"
ALIGN_STAGE = "bottleneck"
AMP = "auto"  #@param ["auto", "off", "bf16", "fp16"]

assert N_SCENES > 0 and EPOCHS > 0 and BATCH_SIZE > 0
assert 0 < VAL_SPLIT < 1


In [ ]:
# 2. Mount Drive, clone/pull đúng branch và cài dependencies
import os
import shutil
import subprocess
import sys
from pathlib import Path

from google.colab import drive
drive.mount("/content/drive")

def run(cmd, *, cwd=None, env=None):
    cmd = [str(x) for x in cmd]
    print("$", " ".join(cmd), flush=True)
    subprocess.run(cmd, cwd=cwd, env=env, check=True)

repo = Path(LOCAL_REPO)
if repo.exists() and not (repo / ".git").is_dir():
    raise RuntimeError(f"{repo} đã tồn tại nhưng không phải Git repo")
if not repo.exists():
    run(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, repo])
else:
    # Mỗi lần chạy cell này đều lấy code mới nhất; local edits sẽ làm pull dừng an toàn.
    run(["git", "fetch", "origin", BRANCH], cwd=repo)
    run(["git", "checkout", BRANCH], cwd=repo)
    run(["git", "pull", "--ff-only", "origin", BRANCH], cwd=repo)

os.chdir(repo)
Path(DRIVE_ROOT).mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(Path(DRIVE_ROOT) / "huggingface")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
# pyrealsense2 chỉ dành cho camera robot, không cần cho train và dễ thiếu wheel trên Colab.
requirements = [x.strip() for x in Path("requirements.txt").read_text().splitlines()
                if x.strip() and not x.strip().startswith("pyrealsense2")]
run([sys.executable, "-m", "pip", "install", "-q", *requirements])

gpu = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True, text=True, check=False,
).stdout.strip()
print("branch :", subprocess.check_output(["git", "branch", "--show-current"], text=True).strip())
print("commit :", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip())
print("GPU    :", gpu or "không tìm thấy — hãy chọn Runtime > Change runtime type > GPU")
if not gpu:
    raise RuntimeError("Notebook train yêu cầu GPU")


In [ ]:
# 3. Dựng data lần đầu; các lần sau bỏ qua download và dùng lại từ Drive
cache_key = f"ga-pp-{N_SCENES}-scenes-{'packed' if PACK_MASKS else 'raw'}"
persistent_root = Path(DRIVE_ROOT)
DATA_DIR = persistent_root / "data" / cache_key
SPLIT_DIR = persistent_root / "splits" / cache_key
LOG_DIR = persistent_root / "logs"
ARCHIVES_DIR = persistent_root / "archives" if KEEP_ARCHIVES else DATA_DIR / "_archives"
BUILD_DONE = DATA_DIR / ".build_complete"

required_dirs = ["image", "grasp_instructions", "grasp_label_positive",
                 "part_mask", "scene_description"]
cache_ready = BUILD_DONE.is_file() and all((DATA_DIR / x).is_dir() for x in required_dirs)

if cache_ready:
    print(f"Dùng lại dataset đã có: {DATA_DIR}")
else:
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    cmd = [sys.executable, "script/build_ga_pp_subset.py",
           "--out", DATA_DIR, "--zips-dir", ARCHIVES_DIR,
           "--scenes", N_SCENES, "--workers", DOWNLOAD_WORKERS]
    if PACK_MASKS:
        cmd.append("--pack-masks")
    if IMAGES_FROM_ZIP:
        cmd.append("--images-from-zip")
    if KEEP_ARCHIVES:
        cmd.append("--keep-zips")
    run(cmd)
    BUILD_DONE.write_text(f"branch={BRANCH}\nscenes={N_SCENES}\n")

seen_file, unseen_file = SPLIT_DIR / "seen.obj", SPLIT_DIR / "unseen.obj"
if seen_file.is_file() and unseen_file.is_file():
    print(f"Dùng lại split đã có: {SPLIT_DIR}")
else:
    SPLIT_DIR.mkdir(parents=True, exist_ok=True)
    run([sys.executable, "split/build_grasp_anything_pp.py",
         "--data-dir", DATA_DIR, "--out-dir", SPLIT_DIR])

n_samples = len(list((DATA_DIR / "grasp_label_positive").glob("*.pt")))
free_gb = shutil.disk_usage(persistent_root).free / 1024**3
print(f"Dataset: {n_samples:,} sample | Drive còn trống: {free_gb:.1f} GiB")


In [ ]:
# 4. Train; log và checkpoint được ghi thẳng lên Drive
LOG_DIR.mkdir(parents=True, exist_ok=True)
train_cmd = [
    sys.executable, "train_network.py",
    "--dataset", "grasp-anything-pp",
    "--dataset-path", DATA_DIR,
    "--split-path", SPLIT_DIR,
    "--network", "grconvnet3_align",
    "--use-depth", 0, "--use-rgb", 1, "--seen", 1,
    "--input-size", INPUT_SIZE, "--split", VAL_SPLIT,
    "--epochs", EPOCHS, "--batches-per-epoch", BATCHES_PER_EPOCH,
    "--batch-size", BATCH_SIZE, "--val-batch-size", VAL_BATCH_SIZE,
    "--num-workers", NUM_WORKERS,
    "--lr", LEARNING_RATE, "--lr-schedule", "cosine",
    "--amp", AMP, "--channels-last", "auto",
    "--use-text", 1, "--w-align", W_ALIGN,
    "--align-mode", ALIGN_MODE, "--region-text", REGION_TEXT,
    "--fusion", FUSION, "--align-stage", ALIGN_STAGE,
    "--warmup-epochs", WARMUP_EPOCHS,
    "--diag-interval", 500, "--probe-samples", 8,
    "--counterfactual-every", 5,
    "--logdir", LOG_DIR, "--description", DESCRIPTION,
]

budget = EPOCHS * BATCHES_PER_EPOCH * BATCH_SIZE
print(f"Train budget: {EPOCHS} x {BATCHES_PER_EPOCH} x {BATCH_SIZE} = {budget:,} sample")
run(train_cmd)


In [ ]:
# 5. In checkpoint tốt nhất của lần train gần nhất
import re

runs = sorted((p for p in LOG_DIR.glob(f"*{DESCRIPTION}*") if p.is_dir()),
              key=lambda p: p.stat().st_mtime)
if not runs:
    raise FileNotFoundError(f"Không thấy run {DESCRIPTION!r} trong {LOG_DIR}")
checkpoints = list(runs[-1].glob("epoch_*"))
if not checkpoints:
    raise FileNotFoundError(f"Không thấy checkpoint trong {runs[-1]}")

def checkpoint_iou(path):
    match = re.search(r"iou_([0-9.]+)$", path.name)
    return float(match.group(1)) if match else -1.0

best = max(checkpoints, key=checkpoint_iou)
print("Run       :", runs[-1])
print("Checkpoint:", best)
print("Val IoU   :", checkpoint_iou(best))


## Lưu ý

- Không xoá file `.build_complete` trong thư mục dataset trên Drive nếu muốn tái sử dụng cache. Nếu lần tải đầu bị ngắt, chỉ cần chạy lại cell data; downloader sẽ tiếp tục và bỏ qua các file đã hoàn tất.
- `IMAGES_FROM_ZIP=False` hợp lý cho subset nhỏ. Với trên khoảng 40.000 scene, đặt thành `True`; lần đầu cần thêm khoảng 65 GB để chứa archive ảnh.
- Đổi `N_SCENES` hoặc `PACK_MASKS` tự động tạo cache riêng, không trộn lẫn dataset cũ.
- Train trực tiếp từ nhiều file nhỏ trên Google Drive có thể nghẽn I/O. Nếu GPU thường xuyên rỗi, tăng `NUM_WORKERS` vừa phải hoặc chép cache sang SSD local trước khi train.